In [ ]:
%load_ext autoreload
%autoreload 2

import os 
import sys

import pandas as pd
import numpy as np

from Preproces_prod2 import *
    
local_path = os.getcwd()
code_root = os.path.abspath(os.path.join(local_path, '..', 'Code'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)
code_root = os.path.abspath(os.path.join(local_path, '../..', 'inv'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)

from matching_case_control import call_data,analyze_vrs_data, integer_programming_matching_gurobi,match_nn_max_dist_weigths,comparar_medias_test, charly_mip, charly_double_mip, mylogit, results_match, tabla_marcel, tabla_final

import inv

from IPython.core.display import display
display.max_output_lines = 500  # Adjust the number as needed
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

np.random.seed(42)

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

df_vrs_match_case = pd.read_csv(path_data/'df_vrs_match_case_s39_2012.csv', index_col=0)
df_upc_match_case = pd.read_csv(path_data/'df_upc_match_case_s39_2012.csv', index_col=0)

icd10_codes_fr = [
    "N390",   # Infección del tracto urinario (sin síntomas respiratorios) INFECCION DE VÍAS URINARIAS, SITIO NO ESPECIFICADO
    "A090",     # Gastroenteritis aguda (sin síntomas respiratorios) OTRAS GASTROENTERITIS Y COLITIS NO ESPECIFICADAS DE ORIGEN INFECCIOSO
    "A099",     # Gastroenteritis aguda (sin síntomas respiratorios) GASTROENTERITIS Y COLITIS DE ORIGEN NO ESPECIFICADO
    "A09X",     # Gastroenteritis aguda (sin síntomas respiratorios) DIARREA Y GASTROENTERITIS DE PRESUNO ORIGEN INFECCIOSO
    "R100",  # Cólico infantil (sin fiebre ni síntomas respiratorios) ABDOMEN AGUDO
    "R101",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR ABDOMINAL LOCALIZADO EN PARTE SUPERIOR
    "R102",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR PELVICO Y PERINEAL
    "R103",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR LOCALIZADO EN OTRAS PARTES INFERIORES DEL ABDOMEN
    "R104",  # Cólico infantil (sin fiebre ni síntomas respiratorios) OTROS DOLORES ABDOMINALES Y LOS NO ESPECIFICADOS
    "R11X",  # Cólico infantil (sin fiebre ni síntomas respiratorios) NAUSEA Y VOMITO
    "R634",   # Pérdida de peso (sin fiebre ni síntomas respiratorios) PERDIDA ANORMAL DE PESO
    "R633",   # Dificultades de alimentación (sin fiebre ni síntomas respiratorios) DIFICULTADES Y MALA ADMINISTRACION DE LA ALIMENTACION
    "P599",   # Ictericia neonatal (sin fiebre ni síntomas respiratorios) ICTERICIA NEONATAL, NO ESPECIFICADA
    "R681",  # Llanto anormal (sin fiebre ni síntomas respiratorios) SINTOMAS NO ESPECIFICOS PROPIOS DE LA INFANCIA
    "S099",  # Traumatismo craneal (sin fiebre ni síntomas respiratorios)
    "Z539"    # Cirugía de emergencia (sin fiebre ni síntomas respiratorios)
]

list_experiments=[]
df = (df_vrs_match_case
      .drop(columns=[col for col in df_vrs_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
      .copy()
)


ruts_cariopaticos_1 = pd.read_csv(path_data/'ruts_cardiopatias.csv', index_col=0).query('card1')
ruts_cariopaticos_2 = pd.read_csv(path_data/'ruts_cardiopatias.csv', index_col=0).query('card2')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# 1 FONASA
# 2 ISAPRE
# 3 CAPREDENA
# 4 DIPRECA
# 5 SISA
# 96 NINGUNA
# 99 DESCONOCIDO

df.PREVI.value_counts()

PREVI
1.0     23556
2.0      3551
96.0     1622
99.0      530
4.0       344
5.0       204
3.0       109
Name: count, dtype: int64

In [ ]:
from itertools import product, chain, combinations

# Crear listas de filtros por categoría
filtros_categorias = {
    "prematuro": [f"prematuro == {value}" for value in df["prematuro"].dropna().unique()],
    "group": [f"group == {value}" for value in df["group"].dropna().unique()],
    "previ": [f"PREVI == {value}" for value in [1.0, 2.0, 96.0]],  # Valores fijos
    "ruralidad": [f"is_rural == {value}" for value in df["is_rural"].dropna().unique()],
    "muy_prematuro": [f"muy_prematuro == {value}" for value in df["muy_prematuro"].dropna().unique()]
}

# Función para obtener todas las combinaciones no vacías de las categorías de filtros
def power_set(iterable):
    """Devuelve el conjunto potencia del iterable, excluyendo el vacío"""
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(1, len(s)+1))

# Generar todas las combinaciones posibles de conjuntos de categorías
categorias_combinaciones = list(power_set(filtros_categorias.keys()))

# Generar combinaciones válidas de filtros
filtros_combinados = []
for categorias in categorias_combinaciones:
    # Para cada combinación de categorías, generar todas las combinaciones de filtros dentro de esas categorías
    filtros_en_categoria = [filtros_categorias[categoria] for categoria in categorias]
    for combinacion in product(*filtros_en_categoria):
        filtros_combinados.append(" & ".join(combinacion))

# Imprimir algunas combinaciones para revisar
for i, filtro in enumerate(filtros_combinados):
    print(f"Filtro {i+1}: {filtro}")

In [ ]:
match_vars_distance_nn=['edad_relativa','PESO']
match_vars_exact_nn = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','PREVI','prematuro']

match_vars_distance_IP = ['PESO','edad_relativa','SEMANAS']
match_vars_exact_IP = ['sexo','region','group','PREVI'] #['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','PREVI','prematuro']
weights = {'edad_relativa': 1,'PESO': 0.4} 

df_cardio_1 = df.query('RUN.isin(@ruts_cariopaticos_1.RUN)')
df_cardio_2 = df.query('RUN.isin(@ruts_cariopaticos_2.RUN)')


filtro_comba_dict = {'prematurs': '(prematuro == 1)',
                     'prema_season': '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                     'prema_catch': '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                     
                     'prema_fonasa': '(prematuro == 1)' + ' & ' + '(PREVI == 1.0)',
                     'prema_isapre': '(prematuro == 1)' + ' & ' + '(PREVI == 2.0)',
                     'prema_sin_previ': '(prematuro == 1)' + ' & ' + '(PREVI == 96.0)',
                     
                     'cardiopats_1': 'RUN.isin(@ruts_cariopaticos_1.RUN)',
                     'cardiopats_2': 'RUN.isin(@ruts_cariopaticos_2.RUN)',

                     'cardio_O_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)',
                     'cardio_Y_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)',
                     
                     'cardio_Y_no_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 0)',
                     'prema_Y_no_cardio': '(prematuro == 1)' + ' & ' + '~RUN.isin(@ruts_cariopaticos_1.RUN)',
                     
                     'cardio_O_prema_catch': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                     'cardio_O_prema_season': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                     'cardio_Y_prema_catch': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                     'cardio_Y_prema_season': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                     
                     'rural': 'is_rural == 1'}

list_experiments = results_match(df_estudio=df_vrs_match_case,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP,
                                 match_vars_exact_IP=match_vars_exact_IP,
                                 weights=weights, 
                                 list_experiments = [],
                                 nn=False,
                                 mip=True)


prematurs 226 14689
creacion conjuntos A_i time: 14.048449516296387
creacion variables time: 69.61698007583618
optimize model 1 time: 0.22461867332458496
optimize model 2 time: 0.3119997978210449
matched_data time: 0.3790407180786133
Total cases = 226, Total controls = 14689
Total cases matched is : 206, Total control matched is : 618
ratio: 1:3

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975      0.025
estado_inmunizacion      -1.73276  0.176796    82.320425  90.315914  67.723604

IC disntace: 1.2038522891837267
prema_season 91 7141
creacion conjuntos A_i time: 2.6715848445892334
creacion variables time: 23.7533597946167
optimize model 1 time: 0.027940988540649414
optimize model 2 time: 0.06247711181640625
matched_data time: 0.15173006057739258
Total cases = 91, Total controls = 7141
Total cases matched is : 85, Total control matched is : 255
ratio: 1:3

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectivida

In [ ]:
##################################################################################################### PREMATUROS_ANTES_DE_FUNCTION ########################################################
    df = df_vrs_match_case.query(filtro).copy()
    df_case = df.query("diag_vrs==True").copy()
    df_control = df.query("diag_vrs==False").copy()

    n_casos = df_case.shape[0]
    n_controls = df_control.shape[0]
    
    print(name_filt, n_casos, n_controls )
    
    if n_casos < 1 or n_controls < 1:
        continue
    
    # matched_data, matched_incompleto, matched_controls, cases, controls, ratio, n_control_matched, n_case_matched = match_nn_max_dist_weigths(df_control, df_case,
    #                                                             match_vars_nn= match_vars_distance_nn, 
    #                                                             match_vars_exact = match_vars_exact_nn,
    #                                                             match_vars_onehot=[],
    #                                                             ratio="1:3",
    #                                                             max_distance=5,
    #                                                             weights=weights)

    # results = mylogit(matched_data,cases, controls, ratio, n_control_matched, n_case_matched )
    # if results == 'No hay no inmunizados':
    #     dic_aux = {'filtro': name_filt,
    #             'modelo': 'nn',
    #             'problema': 'No hay no inmunizados'}
    #     list_experiments.append(dic_aux)
    # else:
    #     results['filtro'] = name_filt
    #     results['model'] = 'nn'
    #     list_experiments.append(results.copy())

    matched_data, feasible_controls, cases, controls, ratio, n_control_matched, n_case_matched = charly_double_mip(df_cases=df_case,
                                                        df_control=df_control, 
                                                        distance_vars=match_vars_distance_IP, 
                                                        exact_var_match = match_vars_exact_IP, 
                                                        ratio="1:3")

    results = mylogit(matched_data,cases, controls, ratio, n_control_matched, n_case_matched )
    if results == 'No hay no inmunizados':
        dic_aux = {'filtro': name_filt,
                'modelo': 'MIP',
                'problema': 'No hay no inmunizados'}
        list_experiments.append(dic_aux)
    else:
        results['filtro'] = name_filt
        results['model'] = 'MIP'
        list_experiments.append(results.copy())

In [ ]:
################################################################################# NO PREMATUROS ##############################################################################################################################

#'no_prema_catch': '(prematuro == 0)' + ' & ' + 'group == "CATCH_UP"',
#'no_prema_season': '(prematuro == 0)' + ' & ' + '(group == "SEASONAL")',
#'no_prema_fonasa': 'prematuro == 0' + ' & ' + 'PREVI == 1.0',
#'no_prema_isapre': '(prematuro == 0)' + ' & ' + '(PREVI == 2.0)',
#'no_prema_sin_previ': '(prematuro == 0)' + ' & ' + '(PREVI == 96.0)',

match_vars_distance_nn=['edad_relativa','PESO']
match_vars_exact_nn = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','PREVI','prematuro']

match_vars_distance_IP = ['PESO','edad_relativa','SEMANAS']
match_vars_exact_IP = ['sexo','region','group','PREVI'] 
weights = {'edad_relativa': 1,'PESO': 0.4} 

#list_experiments=[]

filtro_dict_2 = {'not_prema_season': '(prematuro == 0)' + ' & ' + '(group == "SEASONAL")',
                     'not_prema_catch': '(prematuro == 0)' + ' & ' + '(group == "CATCH_UP")',
                     'not_prema_fonasa': '(prematuro == 0)' + ' & ' + '(PREVI == 1.0)',
                     'not_prema_isapre': '(prematuro == 0)' + ' & ' + '(PREVI == 2.0)',
                     'not_prema_sin_previ': '(prematuro == 0)' + ' & ' + '(PREVI == 96.0)',
                    }

# with open("match_prints.txt", "w") as file:
#     sys.stdout = file

for name_filt, filtro in filtro_dict_2.items():
    df = df_vrs_match_case.query(filtro).copy()
    
    df_case = df.query("diag_vrs==True").copy()
    df_control = df.query("diag_vrs==False").copy()

    n_casos = df_case.shape[0]
    n_controls = df_control.shape[0]
    
    print(name_filt, n_casos, n_controls )
    
    if n_casos < 1 or n_controls < 1:
        continue
    
    matched_data, matched_incompleto, matched_controls, cases, controls, ratio, n_control_matched, n_case_matched = match_nn_max_dist_weigths(df_control, df_case,
                                                                match_vars_nn= match_vars_distance_nn, 
                                                                match_vars_exact = match_vars_exact_nn,
                                                                match_vars_onehot=[],
                                                                ratio="1:3",
                                                                max_distance=5,
                                                                weights=weights)

    results = mylogit(matched_data,cases, controls, ratio, n_control_matched, n_case_matched )
    if results == 'No hay no inmunizados':
        dic_aux = {'filtro': name_filt,
                'modelo': 'nn',
                'problema': 'No hay no inmunizados'}
        list_experiments.append(dic_aux)
    else:
        results['filtro'] = name_filt
        results['model'] = 'nn'
        list_experiments.append(results.copy())

    matched_data, feasible_controls, cases, controls, ratio, n_control_matched, n_case_matched = charly_double_mip(df_cases=df_case,
                                                        df_control=df_control, 
                                                        distance_vars=match_vars_distance_IP, 
                                                        exact_var_match = match_vars_exact_IP, 
                                                        ratio="1:3")

    results = mylogit(matched_data,cases, controls, ratio, n_control_matched, n_case_matched )
    if results == 'No hay no inmunizados':
        dic_aux = {'filtro': name_filt,
                'modelo': 'MIP',
                'problema': 'No hay no inmunizados'}
        list_experiments.append(dic_aux)
    else:
        results['filtro'] = name_filt
        results['model'] = 'MIP'
        list_experiments.append(results.copy())

In [ ]:
################################################################################# COMUNAS ##############################################################################################################################
match_vars_distance_nn=['edad_relativa','PESO']
match_vars_exact_nn = ['SEXO','SEMANAS','group','muy_prematuro','PREVI','prematuro']

match_vars_distance_IP = ['PESO','edad_relativa','SEMANAS']
match_vars_exact_IP = ['sexo','region','group','PREVI'] 
weights = {'edad_relativa': 1,'PESO': 0.4} 

#list_experiments=[]

comunas_eventuables = df_vrs_match_case.query('event==1').query('region=="METROPOLITANA"').COMUNA.value_counts().reset_index().query('count>=10').COMUNA.unique()
df_comunas = pd.read_excel(path_data/'Comunas, Provincias y Regiones 2008.xls')
df_comunas.columns = ['code', 'COMUNA', 'PROVINCIA', 'REGION']
df_comunas_use = df_comunas.query('code.isin(@comunas_eventuables)')[['code','COMUNA']]

for comuna_filt in comunas_eventuables:
    df = df_vrs_match_case.query('region=="METROPOLITANA"').query('COMUNA==@comuna_filt').copy()
    df_case = df.query("diag_vrs==True").copy()
    df_control = df.query("diag_vrs==False").copy()

    n_casos = df_case.shape[0]
    n_controls = df_control.shape[0]
    
    print(comuna_filt, n_casos, n_controls )
    
    if n_casos < 1 or n_controls < 1:
        continue
    
    matched_data, matched_incompleto, matched_controls, cases, controls, ratio, n_control_matched, n_case_matched = match_nn_max_dist_weigths(df_control, df_case,
                                                                match_vars_nn= match_vars_distance_nn, 
                                                                match_vars_exact = match_vars_exact_nn,
                                                                match_vars_onehot=[],
                                                                ratio="1:3",
                                                                max_distance=5,
                                                                weights=weights)

    results = mylogit(matched_data,cases, controls, ratio, n_control_matched, n_case_matched )
    if results == 'No hay no inmunizados':
        dic_aux = {'filtro': df_comunas_use.query('code==@comuna_filt').COMUNA.values[0],
                'modelo': 'nn',
                'problema': 'No hay no inmunizados'}
        list_experiments.append(dic_aux)
    else:
        results['filtro'] = df_comunas_use.query('code==@comuna_filt').COMUNA.values[0]
        results['model'] = 'nn'
        list_experiments.append(results.copy())

    matched_data, feasible_controls, cases, controls, ratio, n_control_matched, n_case_matched = charly_double_mip(df_cases=df_case,
                                                        df_control=df_control, 
                                                        distance_vars=match_vars_distance_IP, 
                                                        exact_var_match = match_vars_exact_IP, 
                                                        ratio="1:3")

    results = mylogit(matched_data,cases, controls, ratio, n_control_matched, n_case_matched )
    if results == 'No hay no inmunizados':
        dic_aux = {'filtro': df_comunas_use.query('code==@comuna_filt').COMUNA.values[0],
                'modelo': 'MIP',
                'problema': 'No hay no inmunizados'}
        list_experiments.append(dic_aux)
    else:
        results['filtro'] = df_comunas_use.query('code==@comuna_filt').COMUNA.values[0]
        results['model'] = 'MIP'
        list_experiments.append(results.copy())

In [ ]:
################################################################################## TABLA MARCEL A SECADS ####################################################################################################################
experiment_dfs = []
list_experiments_edit = [dic for dic in list_experiments if len(dic) != 3]

for result in list_experiments_edit:
    # Obtener el DataFrame de resultados
    results_df_transposed =(
        result['results_df'].T
        .assign(
            efectividad = lambda df : np.where(df["0.025 Conf Interval"].values[0].round(2) < -100, 
                                              'na',
                                              f'{str(df["Efectividad"].values[0].round(2))} ({str(df["0.025 Conf Interval"].values[0].round(2))}; {str(df["0.975 Conf Interval"].values[0].round(2))})'),        
            #pseudo_p_value = lambda df : check_signs(df['0.025 Conf Interval'], df['0.975 Conf Interval'])
        )
        .assign(
                total_cases = result['total_cases'],
                total_controls = result['total_controls'],
                cases_matched = result['cases_matched'],
                controls_matched = result['controls_matched'],
                ratio = result['ratio'],
                method = result['model'],
                filtro = result['filtro']
            )
        .pipe(lambda df:df.set_index(['filtro'])) #.pipe(lambda df:df.set_index(['filtro','method']))
        ) 
    
    # Añadir el DataFrame a la lista
    experiment_dfs.append(results_df_transposed)

# Concatenar todos los DataFrames en uno solo
all_results_df = pd.concat(experiment_dfs, ignore_index=False)

# Mostrar el DataFrame concatenado
#display(all_results_df)




marcel_summary= (all_results_df[['efectividad']].reset_index()
#.reset_index()
#.unstack(-1)
#.unstack(-2)
#.unstack(-1)
)

display(marcel_summary)



In [130]:
def tabla_marcel(list_experiments):
    
    experiment_dfs = []
    list_experiments_edit = [dic for dic in list_experiments if len(dic) != 3]

    for result in list_experiments_edit:
        results_df_transposed =(
            result['results_df'].T
            .assign(
                efectividad = lambda df : np.where(df["0.025 Conf Interval"].values[0].round(2) < -100, 
                                                'na',
                                                f'{str(df["Efectividad"].values[0].round(2))} ({str(df["0.025 Conf Interval"].values[0].round(2))}; {str(df["0.975 Conf Interval"].values[0].round(2))})'),        
                #pseudo_p_value = lambda df : check_signs(df['0.025 Conf Interval'], df['0.975 Conf Interval'])
            )
            .assign(
                    total_cases = result['total_cases'],
                    total_controls = result['total_controls'],
                    cases_matched = result['cases_matched'],
                    controls_matched = result['controls_matched'],
                    ratio = result['ratio'],
                    method = result['model'],
                    filtro = result['filtro']
                )
            .pipe(lambda df:df.set_index(['filtro'])) 
            ) 
        
        experiment_dfs.append(results_df_transposed)

    all_results_df = pd.concat(experiment_dfs, ignore_index=False)

    marcel_summary= (all_results_df[['efectividad']].reset_index()
    )
    
    return marcel_summary, all_results_df

marcel_summary, all_results_df = tabla_marcel(list_experiments)
display(marcel_summary)

,filtro,efectividad
0,prematurs,82.32 (67.72; 90.32)
1,prema_season,na
2,prema_catch,84.97 (71.19; 92.16)
3,prema_fonasa,80.6 (64.88; 89.28)
4,prema_isapre,na
5,cardiopats_1,86.67 (31.28; 97.41)
6,cardiopats_2,na
7,cardio_O_prema,82.93 (69.98; 90.3)
8,cardio_Y_prema,na
9,cardio_Y_no_prema,91.66 (25.43; 99.07)


In [ ]:
#################################################################################################### ene_match a secas ############################################################################################################
# aver = marcel_summary.reset_index()
# #aver.columns = ['method','filtro','Efectividad_MIP','Efectividad_NN']
# aver.columns = ['filtro','Efectividad_MIP','Efectividad_NN']
# aver.columns = ['filtro','Efectividad_MIP','Efectividad_NN']
# aver2 = aver.sort_values(by=['Efectividad_MIP'], ascending=False) #.set_index

In [ ]:
enes_matches, all_n_eff  = tabla_final(all_results_df,marcel_summary)

In [55]:
list_experiments_copy = list_experiments.copy()